# 09 — Evaluación de Utilidad Analítica

**Fase 3 — Evaluación [OE 4]**  
Valida si los datos sintéticos mantienen el valor predictivo de los datos reales para tareas clínicas *downstream*.

Experimentos implementados:
1. **TRTR** (Train on Real, Test on Real) — línea base de rendimiento con datos reales
2. **TSTR** (Train on Synthetic, Test on Real) — degradación al entrenar sobre datos sintéticos
3. **Data augmentation** — 50 % real + 50 % sintético vs. solo real
4. **Sepsis** (tarea secundaria) — detección de sepsis mediante proxy qSOFA

Clasificadores *downstream*: Regresión Logística · Random Forest · XGBoost · MLP  
Métricas: **AUROC · AUPRC · F1** (elección justificada por desbalance de clases 86/14 %)  
Tarea principal: predicción de mortalidad intrahospitalaria (`hospital_expire_flag`)

## 0. Imports y configuración

In [ ]:
import sys, subprocess, warnings
warnings.filterwarnings('ignore')

for pkg, imp in [("xgboost", "xgboost")]:
    try:
        __import__(imp)
    except ImportError:
        print(f"Instalando {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb

ROOT      = Path('..')
PROCESSED = ROOT / 'data' / 'processed'
SYNTHETIC = ROOT / 'data' / 'synthetic'
REPORTS   = ROOT / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)

np.random.seed(42)
sns.set_theme(style='whitegrid', palette='muted')

COLORS = {
    'TRTR':          'dimgray',
    'CTGAN':         'tomato',
    'TVAE':          'seagreen',
    'TabDDPM':       'steelblue',
    'DP-CTGAN ε=∞':  'mediumpurple',
    'DP-CTGAN ε=10': 'darkorchid',
    'DP-CTGAN ε=5':  'purple',
    'DP-CTGAN ε=1':  'indigo',
}

print('Entorno listo.')

## 1. Carga de datos

In [ ]:
real   = pd.read_parquet(PROCESSED / 'tabular_48h.parquet')
TARGET = 'hospital_expire_flag'
id_cols = [c for c in real.columns if c.endswith('_id')]
feat_cols = [c for c in real.columns
             if c not in id_cols + [TARGET] and real[c].dtype != object]

print(f'Dataset real : {real.shape}  |  mortalidad: {real[TARGET].mean()*100:.1f}%')
print(f'Features     : {len(feat_cols)}')
print(f'ID cols      : {id_cols}')

SYNTH_FILES = {
    'CTGAN':         SYNTHETIC / 'ctgan_samples.parquet',
    'TVAE':          SYNTHETIC / 'tvae_samples.parquet',
    'TabDDPM':       SYNTHETIC / 'tabddpm_samples.parquet',
    'DP-CTGAN ε=∞':  SYNTHETIC / 'dp_ctgan_εinf_samples.parquet',
    'DP-CTGAN ε=10': SYNTHETIC / 'dp_ctgan_ε10_samples.parquet',
    'DP-CTGAN ε=5':  SYNTHETIC / 'dp_ctgan_ε5_samples.parquet',
    'DP-CTGAN ε=1':  SYNTHETIC / 'dp_ctgan_ε1_samples.parquet',
}

synth = {}
synth_has_target = {}
print()
for name, path in SYNTH_FILES.items():
    if path.exists():
        df = pd.read_parquet(path)
        synth[name] = df
        has_target = TARGET in df.columns
        synth_has_target[name] = has_target
        if has_target:
            tag = f"mortalidad: {df[TARGET].mean()*100:.1f}%"
        else:
            tag = "sin TARGET — TSTR mortalidad no disponible"
        print(f'  [OK] {name:<20} {df.shape}  {tag}')
    else:
        print(f'  [--] {name:<20} (archivo no encontrado)')

tstr_models = [m for m, has in synth_has_target.items() if has]
print(f'\nModelos disponibles para TSTR mortalidad: {tstr_models}')
if len(tstr_models) < len(synth):
    skipped = [m for m, has in synth_has_target.items() if not has]
    print(f'⚠  Modelos sin TARGET (omitidos en TSTR): {skipped}')
    print('   Para incluirlos, regenerar con nb07 añadiendo hospital_expire_flag al dataset de entrenamiento.')

## 2. Setup: features, partición fija y clasificadores downstream

In [ ]:
# Partición fija 80/20 del dataset real — el test set es el mismo en TRTR y TSTR
X_real = real[feat_cols].fillna(0).values.astype(np.float32)
y_real = real[TARGET].values.astype(int)

X_train_r, X_test, y_train_r, y_test = train_test_split(
    X_real, y_real, test_size=0.2, stratify=y_real, random_state=42
)

pos_weight_real = (y_train_r == 0).sum() / (y_train_r == 1).sum()

print(f'Train real : {X_train_r.shape}  |  mortalidad: {y_train_r.mean()*100:.1f}%')
print(f'Test real  : {X_test.shape}    |  mortalidad: {y_test.mean()*100:.1f}%')
print(f'scale_pos_weight (XGBoost): {pos_weight_real:.2f}')

# ---------------------------------------------------------------

def get_classifiers(pos_weight):
    """Devuelve los 4 clasificadores downstream con manejo de desbalance."""
    return {
        'LogReg': LogisticRegression(
            max_iter=1000, class_weight='balanced', C=1.0,
            solver='lbfgs', random_state=42
        ),
        'RandomForest': RandomForestClassifier(
            n_estimators=200, class_weight='balanced',
            n_jobs=-1, random_state=42
        ),
        'XGBoost': xgb.XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=pos_weight,
            eval_metric='logloss', random_state=42, verbosity=0
        ),
        'MLP': MLPClassifier(
            hidden_layer_sizes=(128, 64), activation='relu',
            max_iter=300, random_state=42,
            early_stopping=True, validation_fraction=0.1
        ),
    }


def evaluate_clf(clf, X_tr, y_tr, X_te, y_te, scale=False, sample_weights=None):
    """Entrena clf en (X_tr, y_tr) y evalúa en (X_te, y_te).
    Si scale=True aplica StandardScaler ajustado sobre X_tr.
    sample_weights se pasa via sample_weight si el clasificador lo soporta
    (MLPClassifier no lo soporta — fallback silencioso).
    Devuelve dict con auroc, auprc, f1.
    """
    if scale:
        sc = StandardScaler()
        X_tr = sc.fit_transform(X_tr)
        X_te = sc.transform(X_te)

    if sample_weights is not None:
        try:
            clf.fit(X_tr, y_tr, sample_weight=sample_weights)
        except TypeError:
            # MLPClassifier no acepta sample_weight — entrena sin ponderación
            clf.fit(X_tr, y_tr)
    else:
        clf.fit(X_tr, y_tr)

    y_prob = clf.predict_proba(X_te)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    return {
        'auroc': roc_auc_score(y_te, y_prob),
        'auprc': average_precision_score(y_te, y_prob),
        'f1':    f1_score(y_te, y_pred, zero_division=0),
    }


def synth_feat_matrix(df_s, feat_cols):
    """Construye matriz de features para un dataset sintético,
    rellenando con 0 las columnas ausentes para mantener dimensionalidad.
    """
    cols_s = [c for c in feat_cols if c in df_s.columns]
    X_s = df_s[cols_s].fillna(0).values.astype(np.float32)
    if len(cols_s) < len(feat_cols):
        X_full = np.zeros((len(X_s), len(feat_cols)), dtype=np.float32)
        idx = [feat_cols.index(c) for c in cols_s]
        X_full[:, idx] = X_s
        return X_full
    return X_s


print('Helpers definidos.')

## 3. TRTR — Línea base (Train on Real, Test on Real)

In [ ]:
print('=== TRTR: Train on Real, Test on Real ===\n')

trtr_results = {}
for clf_name, clf in get_classifiers(pos_weight_real).items():
    scale = clf_name in ('LogReg', 'MLP')
    sw    = compute_sample_weight('balanced', y_train_r) if clf_name == 'MLP' else None

    res = evaluate_clf(clf, X_train_r, y_train_r, X_test, y_test,
                       scale=scale, sample_weights=sw)
    trtr_results[clf_name] = res
    print(f'  {clf_name:<15}  AUROC={res["auroc"]:.4f}  AUPRC={res["auprc"]:.4f}  F1={res["f1"]:.4f}')

trtr_df = pd.DataFrame(trtr_results).T
print(f'\n  {"MEDIA":<15}  '
      f'AUROC={trtr_df["auroc"].mean():.4f}  '
      f'AUPRC={trtr_df["auprc"].mean():.4f}  '
      f'F1={trtr_df["f1"].mean():.4f}')

## 4. TSTR — Train on Synthetic, Test on Real

Para cada modelo con `hospital_expire_flag` disponible:
se entrena cada clasificador sobre el dataset sintético completo (N = 22 520)
y se evalúa sobre el mismo test set real fijo del paso anterior.

In [ ]:
print('=== TSTR: Train on Synthetic, Test on Real ===\n')

tstr_all = {}

for model_name in tstr_models:
    df_s = synth[model_name]
    X_s  = synth_feat_matrix(df_s, feat_cols)
    y_s  = df_s[TARGET].values.astype(int)

    pos_w_s = (y_s == 0).sum() / max((y_s == 1).sum(), 1)
    clfs_s  = get_classifiers(pos_w_s)

    print(f'  [{model_name}]  mortalidad sintética: {y_s.mean()*100:.1f}%')
    tstr_all[model_name] = {}

    for clf_name, clf in clfs_s.items():
        scale = clf_name in ('LogReg', 'MLP')
        sw    = compute_sample_weight('balanced', y_s) if clf_name == 'MLP' else None

        res = evaluate_clf(clf, X_s, y_s, X_test, y_test,
                           scale=scale, sample_weights=sw)
        tstr_all[model_name][clf_name] = res
        trtr_auroc = trtr_results[clf_name]['auroc']
        delta = res['auroc'] - trtr_auroc
        print(f'    {clf_name:<15}  AUROC={res["auroc"]:.4f}  AUPRC={res["auprc"]:.4f}  '
              f'F1={res["f1"]:.4f}  ΔAUROC={delta:+.4f}')
    print()

In [ ]:
# Tabla de gaps TRTR → TSTR media por modelo (promedio de 4 clasificadores)
print('=== GAP TRTR → TSTR (promedio de 4 clasificadores) ===\n')

gap_rows = []
for model_name, clf_res in tstr_all.items():
    for clf_name, res in clf_res.items():
        gap_rows.append({
            'Modelo':       model_name,
            'Clasificador': clf_name,
            'TRTR_AUROC':   trtr_results[clf_name]['auroc'],
            'TSTR_AUROC':   res['auroc'],
            'ΔAUROC':       res['auroc'] - trtr_results[clf_name]['auroc'],
            'TSTR_AUPRC':   res['auprc'],
            'TSTR_F1':      res['f1'],
        })

gap_df = pd.DataFrame(gap_rows)

pivot = gap_df.groupby('Modelo')[['TSTR_AUROC', 'TSTR_AUPRC', 'TSTR_F1', 'ΔAUROC']].mean()
pivot.columns = ['AUROC_medio', 'AUPRC_medio', 'F1_medio', 'ΔAUROC_medio']
trtr_ref = trtr_df.mean()
print(f"  {'TRTR (baseline)':<22}  AUROC={trtr_ref['auroc']:.4f}  "
      f"AUPRC={trtr_ref['auprc']:.4f}  F1={trtr_ref['f1']:.4f}\n")
print(pivot.round(4).to_string())

In [ ]:
# Figura 1: AUROC TSTR por modelo × clasificador (heatmap) + gap respecto TRTR (barras)
tstr_auroc_matrix = pd.DataFrame({
    m: {c: r['auroc'] for c, r in clf_res.items()}
    for m, clf_res in tstr_all.items()
}).T

fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(tstr_models) * 0.8 + 2)))

trtr_max_auroc = trtr_df['auroc'].max()
sns.heatmap(
    tstr_auroc_matrix, annot=True, fmt='.3f', cmap='RdYlGn',
    vmin=0.45, vmax=min(trtr_max_auroc + 0.03, 1.0),
    ax=axes[0], linewidths=0.5, cbar=True
)
axes[0].set_title('AUROC TSTR por modelo × clasificador')
axes[0].set_xlabel('')
axes[0].set_ylabel('')

avg_tstr  = tstr_auroc_matrix.mean(axis=1).sort_values(ascending=False)
trtr_avg  = trtr_df['auroc'].mean()
ax2 = axes[1]
bars = ax2.barh(
    range(len(avg_tstr)), avg_tstr.values,
    color=[COLORS.get(m, 'steelblue') for m in avg_tstr.index],
    alpha=0.85
)
ax2.axvline(trtr_avg, color='dimgray', linestyle='--', lw=1.5,
            label=f'TRTR medio ({trtr_avg:.3f})')
ax2.set_yticks(range(len(avg_tstr)))
ax2.set_yticklabels(avg_tstr.index)
ax2.set_xlabel('AUROC medio (4 clasificadores)')
ax2.set_title('TSTR medio vs línea base TRTR')
ax2.legend(fontsize=9)
ax2.set_xlim(0.3, 1.0)
for i, v in enumerate(avg_tstr.values):
    ax2.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(REPORTS / 'utilidad_tstr_auroc.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: reports/utilidad_tstr_auroc.png')

## 5. Data augmentation clínico (50 % real + 50 % sintético)

Se añaden N muestras sintéticas (N = tamaño del train real) al conjunto de entrenamiento real.  
Objetivo: determinar si los datos sintéticos pueden complementar datos reales limitados.

In [ ]:
print('=== Data augmentation: train_real + N muestras sintéticas ===\n')

aug_results = {}
rng = np.random.default_rng(42)

for model_name in tstr_models:
    df_s = synth[model_name]
    X_s  = synth_feat_matrix(df_s, feat_cols)
    y_s  = df_s[TARGET].values.astype(int)

    n_add = len(X_train_r)
    replace = n_add > len(X_s)
    idx_s = rng.choice(len(X_s), n_add, replace=replace)

    X_aug = np.vstack([X_train_r, X_s[idx_s]])
    y_aug = np.concatenate([y_train_r, y_s[idx_s]])

    pos_w_aug = (y_aug == 0).sum() / max((y_aug == 1).sum(), 1)
    clfs_aug  = get_classifiers(pos_w_aug)

    print(f'  [{model_name}]  aug size={len(X_aug)}  mortalidad aug: {y_aug.mean()*100:.1f}%')
    aug_results[model_name] = {}

    for clf_name, clf in clfs_aug.items():
        scale = clf_name in ('LogReg', 'MLP')
        sw    = compute_sample_weight('balanced', y_aug) if clf_name == 'MLP' else None

        res = evaluate_clf(clf, X_aug, y_aug, X_test, y_test,
                           scale=scale, sample_weights=sw)
        aug_results[model_name][clf_name] = res

    aug_auroc = np.mean([r['auroc'] for r in aug_results[model_name].values()])
    delta_aug = aug_auroc - trtr_df['auroc'].mean()
    tstr_auroc_m = np.mean([tstr_all[model_name][c]['auroc'] for c in tstr_all[model_name]])
    print(f'    AUROC aug medio={aug_auroc:.4f}  ΔAUROC vs TRTR={delta_aug:+.4f}  '
          f'vs TSTR puro={aug_auroc - tstr_auroc_m:+.4f}')
    print()

## 6. Tarea secundaria: detección de sepsis

Se define un *proxy* de sepsis usando el criterio **qSOFA** sobre las variables del snapshot de 48h:  
- `resp_rate_mean ≥ 22` (taquipnea)  
- `sbp_mean ≤ 100` mmHg (hipotensión)  
- `gcs_total_mean ≤ 14` (alteración del estado mental)  

Paciente con sepsis: qSOFA ≥ 2 (criterio Sepsis-3, Singer et al. 2016).

Para TSTR en sepsis: el label qSOFA se computa también sobre los features sintéticos,
evaluando si el modelo generativo preserva las correlaciones clínicas que definen el síndrome.

In [ ]:
RESP_COL = 'resp_rate_mean'
SBP_COL  = 'sbp_mean'
GCS_COL  = 'gcs_total_mean'

avail = {c: c in real.columns for c in [RESP_COL, SBP_COL, GCS_COL]}
print('Columnas qSOFA disponibles:', avail)

qsofa_real = pd.Series(0, index=real.index)
if avail[RESP_COL]: qsofa_real += (real[RESP_COL] >= 22).astype(int)
if avail[SBP_COL]:  qsofa_real += (real[SBP_COL]  <= 100).astype(int)
if avail[GCS_COL]:  qsofa_real += (real[GCS_COL]  <= 14).astype(int)

if not all(avail.values()):
    print('⚠  Alguna columna qSOFA ausente — usando componentes disponibles.')

y_sepsis = (qsofa_real >= 2).astype(int).values
prevalencia = y_sepsis.mean() * 100
print(f'Label sepsis (qSOFA ≥ 2): {prevalencia:.1f}% positivos ({y_sepsis.sum():,} pacientes de {len(y_sepsis):,})')

# Correlación sepsis vs mortalidad como sanity check clínico
corr_sep_mort = pd.Series(y_sepsis, index=real.index).corr(real[TARGET])
print(f'Correlación sepsis (qSOFA) vs mortalidad: {corr_sep_mort:.3f}  (positiva esperada)')

In [ ]:
# Partición sepsis — misma semilla y mismos índices de test que en mortalidad
_, _, y_sep_train, y_sep_test = train_test_split(
    X_real, y_sepsis, test_size=0.2, stratify=y_sepsis, random_state=42
)
pos_w_sep_real = (y_sep_train == 0).sum() / max((y_sep_train == 1).sum(), 1)
print(f'Sepsis prevalencia — train: {y_sep_train.mean()*100:.1f}%  test: {y_sep_test.mean()*100:.1f}%\n')

# TRTR sepsis (4 clasificadores)
print('--- TRTR sepsis ---')
sepsis_trtr = {}
for clf_name, clf in get_classifiers(pos_w_sep_real).items():
    scale = clf_name in ('LogReg', 'MLP')
    sw    = compute_sample_weight('balanced', y_sep_train) if clf_name == 'MLP' else None
    res   = evaluate_clf(clf, X_train_r, y_sep_train, X_test, y_sep_test,
                         scale=scale, sample_weights=sw)
    sepsis_trtr[clf_name] = res
    print(f'  {clf_name:<15}  AUROC={res["auroc"]:.4f}  AUPRC={res["auprc"]:.4f}  F1={res["f1"]:.4f}')

sepsis_trtr_avg_auroc = np.mean([r['auroc'] for r in sepsis_trtr.values()])
print(f'  {"MEDIA":<15}  AUROC={sepsis_trtr_avg_auroc:.4f}\n')

# TSTR sepsis — el label se deriva de los features sintéticos (qSOFA sobre datos sintéticos)
# Esto mide si el generador preserva las correlaciones clínicas del síndrome séptico.
print('--- TSTR sepsis (qSOFA derivado de features sintéticos, XGBoost) ---')
sepsis_tstr = {}
for model_name in tstr_models:
    df_s = synth[model_name]
    X_s  = synth_feat_matrix(df_s, feat_cols)

    q_s = pd.Series(0, index=df_s.index)
    if RESP_COL in df_s.columns: q_s += (df_s[RESP_COL] >= 22).astype(int)
    if SBP_COL  in df_s.columns: q_s += (df_s[SBP_COL]  <= 100).astype(int)
    if GCS_COL  in df_s.columns: q_s += (df_s[GCS_COL]  <= 14).astype(int)
    y_sep_s = (q_s >= 2).astype(int).values

    pos_w_s = (y_sep_s == 0).sum() / max((y_sep_s == 1).sum(), 1)
    clf_xgb = xgb.XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=pos_w_s,
        eval_metric='logloss', random_state=42, verbosity=0
    )
    clf_xgb.fit(X_s, y_sep_s)
    y_prob = clf_xgb.predict_proba(X_test)[:, 1]
    auroc  = roc_auc_score(y_sep_test, y_prob)
    auprc  = average_precision_score(y_sep_test, y_prob)

    sepsis_tstr[model_name] = {'auroc': auroc, 'auprc': auprc}
    delta = auroc - sepsis_trtr_avg_auroc
    print(f'  {model_name:<20}  AUROC={auroc:.4f}  AUPRC={auprc:.4f}  '
          f'ΔAUROC={delta:+.4f}  sep. sintético: {y_sep_s.mean()*100:.1f}%')

## 7. Tabla resumen y exportación

In [ ]:
print('=== TABLA RESUMEN DE UTILIDAD ANALÍTICA ===\n')

rows = []

# Fila TRTR
rows.append({
    'Condición':    'TRTR (baseline)',
    'Protocolo':    'TRTR',
    'AUROC':        trtr_df['auroc'].mean(),
    'AUPRC':        trtr_df['auprc'].mean(),
    'F1':           trtr_df['f1'].mean(),
    'ΔAUROC':       0.0,
    'AUROC_sepsis': sepsis_trtr_avg_auroc,
})

# Filas TSTR
for model_name, clf_res in tstr_all.items():
    tstr_auroc = np.mean([r['auroc'] for r in clf_res.values()])
    tstr_auprc = np.mean([r['auprc'] for r in clf_res.values()])
    tstr_f1    = np.mean([r['f1']    for r in clf_res.values()])
    rows.append({
        'Condición':    model_name,
        'Protocolo':    'TSTR',
        'AUROC':        tstr_auroc,
        'AUPRC':        tstr_auprc,
        'F1':           tstr_f1,
        'ΔAUROC':       tstr_auroc - trtr_df['auroc'].mean(),
        'AUROC_sepsis': sepsis_tstr.get(model_name, {}).get('auroc', float('nan')),
    })

# Filas augmentation
for model_name, clf_res in aug_results.items():
    aug_auroc = np.mean([r['auroc'] for r in clf_res.values()])
    aug_auprc = np.mean([r['auprc'] for r in clf_res.values()])
    aug_f1    = np.mean([r['f1']    for r in clf_res.values()])
    rows.append({
        'Condición':    f'{model_name} (aug)',
        'Protocolo':    'TSTR+aug',
        'AUROC':        aug_auroc,
        'AUPRC':        aug_auprc,
        'F1':           aug_f1,
        'ΔAUROC':       aug_auroc - trtr_df['auroc'].mean(),
        'AUROC_sepsis': float('nan'),
    })

utility_df = pd.DataFrame(rows).set_index('Condición')
print(utility_df.round(4).to_string())

utility_df.to_csv(REPORTS / 'utilidad_summary.csv')
gap_df.to_csv(REPORTS / 'utilidad_gap_per_clf.csv', index=False)
print('\nGuardado: reports/utilidad_summary.csv')
print('Guardado: reports/utilidad_gap_per_clf.csv')

In [ ]:
# Figura 2: resumen completo de utilidad
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

trtr_avg_auroc = trtr_df['auroc'].mean()
trtr_avg_auprc = trtr_df['auprc'].mean()

# --- Panel superior izquierdo: AUROC TSTR vs TSTR+aug vs TRTR ---
ax = axes[0, 0]
tstr_rows = utility_df[utility_df['Protocolo'] == 'TSTR']
aug_rows  = utility_df[utility_df['Protocolo'] == 'TSTR+aug']
if not tstr_rows.empty and not aug_rows.empty:
    x = np.arange(len(tstr_rows))
    w = 0.35
    ax.bar(x - w/2, tstr_rows['AUROC'], w, label='TSTR puro',
           color=[COLORS.get(m, 'steelblue') for m in tstr_rows.index], alpha=0.85)
    ax.bar(x + w/2, aug_rows['AUROC'], w, label='TSTR + aug',
           color=[COLORS.get(m.replace(' (aug)', ''), 'steelblue') for m in aug_rows.index],
           alpha=0.5, hatch='//')
    ax.axhline(trtr_avg_auroc, color='dimgray', linestyle='--', lw=1.5,
               label=f'TRTR ({trtr_avg_auroc:.3f})')
    ax.set_xticks(x)
    ax.set_xticklabels(tstr_rows.index, rotation=20, ha='right', fontsize=8)
    ax.set_ylabel('AUROC')
    ax.set_title('AUROC mortalidad: TSTR vs TSTR+aug vs TRTR')
    ax.legend(fontsize=8)
    ax.set_ylim(0.3, 1.02)

# --- Panel superior derecho: heatmap ΔAUROC por modelo × clasificador ---
ax = axes[0, 1]
if not gap_df.empty:
    try:
        gap_pivot = gap_df.pivot(index='Modelo', columns='Clasificador', values='ΔAUROC')
        sns.heatmap(gap_pivot, annot=True, fmt='+.3f', cmap='RdYlGn_r',
                    center=0, vmin=-0.35, vmax=0.05,
                    ax=ax, linewidths=0.5, cbar=True)
        ax.set_title('ΔAUROC (TSTR − TRTR) por modelo × clasificador')
        ax.set_xlabel('')
    except Exception as e:
        ax.text(0.5, 0.5, str(e), transform=ax.transAxes, ha='center')

# --- Panel inferior izquierdo: AUPRC comparado ---
ax = axes[1, 0]
if tstr_all:
    models_list = list(tstr_all.keys())
    auprc_tstr  = [np.mean([r['auprc'] for r in tstr_all[m].values()]) for m in models_list]
    auprc_aug   = [np.mean([r['auprc'] for r in aug_results.get(m, {}).values()])
                   if m in aug_results else float('nan') for m in models_list]
    y_pos = np.arange(len(models_list))
    ax.barh(y_pos - 0.2, auprc_tstr, 0.35, label='TSTR puro',
            color=[COLORS.get(m, 'steelblue') for m in models_list], alpha=0.85)
    ax.barh(y_pos + 0.2, auprc_aug, 0.35, label='TSTR+aug',
            color=[COLORS.get(m, 'steelblue') for m in models_list], alpha=0.5, hatch='//')
    ax.axvline(trtr_avg_auprc, color='dimgray', linestyle='--', lw=1.5,
               label=f'TRTR ({trtr_avg_auprc:.3f})')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(models_list)
    ax.set_xlabel('AUPRC')
    ax.set_title('AUPRC mortalidad: TSTR vs TSTR+aug vs TRTR')
    ax.legend(fontsize=8)

# --- Panel inferior derecho: AUROC sepsis ---
ax = axes[1, 1]
if sepsis_tstr:
    sep_models = list(sepsis_tstr.keys())
    sep_aurocs  = [sepsis_tstr[m]['auroc'] for m in sep_models]
    ax.barh(sep_models, sep_aurocs,
            color=[COLORS.get(m, 'steelblue') for m in sep_models], alpha=0.85)
    ax.axvline(sepsis_trtr_avg_auroc, color='dimgray', linestyle='--', lw=1.5,
               label=f'TRTR sepsis ({sepsis_trtr_avg_auroc:.3f})')
    ax.set_xlabel('AUROC')
    ax.set_title('TSTR detección de sepsis (qSOFA, XGBoost)')
    ax.legend(fontsize=9)
    for i, v in enumerate(sep_aurocs):
        ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=8)

plt.suptitle('Validación de Utilidad Analítica — Mortalidad y Sepsis', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS / 'utilidad_summary_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: reports/utilidad_summary_plot.png')

In [ ]:
# Detalle por clasificador para la memoria
print('=== Detalle TRTR vs TSTR por clasificador ===\n')
detail_rows = []
for clf_name in trtr_results:
    row = {
        'Clasificador': clf_name,
        'TRTR_AUROC': trtr_results[clf_name]['auroc'],
        'TRTR_AUPRC': trtr_results[clf_name]['auprc'],
        'TRTR_F1':    trtr_results[clf_name]['f1'],
    }
    for m in tstr_all:
        row[f'{m}_AUROC'] = tstr_all[m][clf_name]['auroc']
    detail_rows.append(row)

detail_df = pd.DataFrame(detail_rows).set_index('Clasificador')
print(detail_df.round(4).to_string())
detail_df.to_csv(REPORTS / 'utilidad_detail_per_clf.csv')
print('\nGuardado: reports/utilidad_detail_per_clf.csv')

# Exportar tabla LaTeX de la tabla resumen principal
tstr_only = utility_df[utility_df['Protocolo'].isin(['TRTR', 'TSTR'])].copy()
tstr_only = tstr_only[['AUROC', 'AUPRC', 'F1', 'ΔAUROC', 'AUROC_sepsis']]
latex_str = tstr_only.round(4).to_latex(
    caption='Utilidad analítica: AUROC, AUPRC y F1 para predicción de mortalidad '
            'y detección de sepsis (TRTR vs. TSTR, promedio de 4 clasificadores downstream).',
    label='tab:utilidad_summary',
    bold_rows=False,
    escape=True,
    na_rep='—',
)
latex_path = REPORTS / 'utilidad_summary_latex.tex'
latex_path.write_text(latex_str, encoding='utf-8')
print(f'Tabla LaTeX guardada: {latex_path}')
print()
print(latex_str)